# Create Cottrell Scholars Awards (FELLOWSHIP PATTERN, embedded-JSON dashboard)

Research Corporation for Science Advancement (RCSA) is the oldest US
foundation dedicated to advancing science (est. 1912). Its flagship
Cottrell Scholar Award supports early-career faculty in chemistry,
physics, and astronomy at US + Canadian universities and PUIs.

**Source pattern:** the foundation publishes the entire awardee
directory as a JSON array embedded in the `/cottrell-scholars/awardee-dashboard/`
page HTML. The page renders the array client-side as a DataTable, but
we parse the embedded JSON directly — no DataTable AJAX endpoint to
hit, no pagination loop, just one request. **First ingest on the
project to use the inline-DataTable-JSON pattern.**

**Awarding body:** Research Corporation for Science Advancement - F4320306487 (US, DOI 10.13039/100001309)

**Schema choices:**
- One row per (year, scholar) pair. `funder_award_id` = `cottrell-{year}-{last-first}` (the dashboard splits multi-year scholars — e.g. Eric Hudson 2005 @ MIT and Eric Hudson 2012 @ UCLA — into separate records, so a single funder_award_id per award is correct).
- `funder_scheme` = `Cottrell Scholar Award`.
- `funding_type` = `research`.
- `amount` / `currency` = USD 120,000 per scholar, verified verbatim from the foundation's own program page (`/cottrell-scholars/cottrell-scholar-award/`: "Each three-year award provides $120,000"). 100% coverage.
- `start_year` = award year; `end_year` = `start_year + 2` (3-year term).
- `lead_investigator.affiliation.country`: derived from the dashboard's `organization_state` field. Canadian provinces map to 'CA' (program eligibility per RCSA is US + Canada). The dashboard uses 'QB' for Quebec instead of 'QC', which the source script normalizes.
- Additional dashboard fields shipped as metadata in description/funder_scheme context: `institution_type` (R1/R2/PUI/Comp/Foreign/etc.), `discipline` (Chemistry/Physics/Astronomy).

**Coverage scope:** all 612 Cottrell Scholar Award recipients 1994-2026, 33 distinct cohorts. The other Cottrell programs administered by RCSA (Cottrell Impact Award, Cottrell SEED Award, Robert Holland Jr. Award) do NOT have embedded scholar JSON on their respective pages — documented in tracker as a Step 0 follow-up.

**Source:** https://rescorp.org/cottrell-scholars/awardee-dashboard/

**S3 location:** `s3a://openalex-ingest/awards/cottrell_scholars/cottrell_scholars.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cottrell_scholars_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cottrell_scholars/cottrell_scholars.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.cottrell_scholars_raw;

## Step 1.5: Inspect raw + discipline/cohort distribution

Expected: ~612 rows across 33 cohorts (1994-2026), 100% on institution + discipline, 100% USD 120,000 amount. Discipline split ~56% Chemistry / ~33% Physics / ~11% Astronomy. Country US/CA split.

In [ ]:
%sql
DESCRIBE openalex.awards.cottrell_scholars_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cottrell_scholars_raw LIMIT 5;

In [ ]:
%sql
-- Cohort counts. Each year should have 15-30 scholars; 1994 is the first year.
SELECT year, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(discipline) AS has_disc
FROM openalex.awards.cottrell_scholars_raw
GROUP BY year
ORDER BY year;

In [ ]:
%sql
-- Discipline split.
SELECT discipline, COUNT(*) AS rows
FROM openalex.awards.cottrell_scholars_raw
GROUP BY discipline
ORDER BY rows DESC;

In [ ]:
%sql
-- Country split (derived from organization_state).
SELECT country, COUNT(*) AS rows
FROM openalex.awards.cottrell_scholars_raw
GROUP BY country
ORDER BY rows DESC;

## Step 1.6: Fail-fast - verify RCSA funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306487;

## Step 2: Transform to award schema

`display_name` = `Cottrell Scholar - {Name} ({discipline}, {year})`. The source has no per-scholar research title or project description; `description` is a concise summary of the scheme metadata (discipline + institution type).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cottrell_scholars_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306487  -- Research Corporation for Science Advancement
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Cottrell Scholar - ', n.name, ' (', n.discipline, ', ', n.year, ')') AS display_name,
    CONCAT('Cottrell Scholar Award in ', COALESCE(n.discipline, 'science'),
           CASE WHEN n.institution_type IS NOT NULL
                THEN CONCAT(' at ', n.institution, ' (', n.institution_type, ')')
                ELSE CONCAT(' at ', COALESCE(n.institution, '(institution unknown)'))
           END) AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    'Cottrell Scholar Award' AS funder_scheme,
    'cottrell_scholars' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) + 2 AS end_year,  -- 3-year term per program page
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.cottrell_scholars_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 135

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cottrell_scholars' AND priority = 135;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    135 AS priority  -- Cottrell Scholar Award
FROM openalex.awards.cottrell_scholars_awards;

## Step 6: Verification

Expect ~612 rows, 100% amount (all USD 120,000), 100% name + institution + discipline + start_year + end_year + country (US/CA split).

In [ ]:
%sql
SELECT COUNT(*) AS total_cottrell_rows FROM openalex.awards.cottrell_scholars_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.cottrell_scholars_awards;

In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.cottrell_scholars_awards;

In [ ]:
%sql
-- §6.7 amount check. Expect 100% USD 120,000 across all rows.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.cottrell_scholars_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.cottrell_scholars_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Country split.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.cottrell_scholars_awards
GROUP BY country
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.cottrell_scholars_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, start_year, end_year, amount,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.cottrell_scholars_awards
ORDER BY start_year DESC, family
LIMIT 10;